In [1]:
!pip install --quiet requests joblib numpy pandas matplotlib scikit-learn lightgbm earthengine-api catboost

In [3]:
import os
import requests
import joblib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from google.colab import drive
from lightgbm import LGBMClassifier, LGBMRegressor
from sklearn.metrics import mean_squared_error, mean_absolute_error, confusion_matrix

# ==============================================================================
# STEP 1: INITIALIZATION & DATA LOADING
# ==============================================================================
print("[1/5] Mounting Drive and setting up paths...")
drive.mount('/content/drive', force_remount=False)

DATA_PATH = '/content/drive/MyDrive/BTP-COLLAB/data/indore-rainfall-data.csv'
JULY_DATA_DIR = '/content/drive/MyDrive/BTP-COLLAB/data/july_data'
MODEL_DIR = '/content/drive/MyDrive/BTP-COLLAB/Models_new'

os.makedirs(JULY_DATA_DIR, exist_ok=True)
os.makedirs(MODEL_DIR, exist_ok=True)

def preprocess_dataset(df):
    df = df.copy()
    if 'date' in df.columns:
        df['date'] = pd.to_datetime(df['date'], format='%d-%m-%Y', errors='coerce').fillna(
            pd.to_datetime(df['date'], errors='coerce')
        )
        df = df.sort_values('date').reset_index(drop=True)

    df['day_of_year'] = df['date'].dt.dayofyear
    df['sin_day'] = np.sin(2 * np.pi * df['day_of_year'] / 365.25)
    df['cos_day'] = np.cos(2 * np.pi * df['day_of_year'] / 365.25)

    df['dtr'] = df['tmax_degC'] - df['tmin_degC']
    df['temp_humidity_idx'] = df['tmax_degC'] * df['humidity_pct']
    df['dewpoint_spread'] = df['tmax_degC'] - df['dewpoint_degC']
    df['pressure_drop_1d'] = df['surface_pressure_hpa'].shift(1) - df['surface_pressure_hpa']

    base_cols = [
        'tmax_degC', 'tmin_degC', 'humidity_pct', 'radiation_wm2', 'wind_speed_ms',
        'dewpoint_degC', 'surface_pressure_hpa', 'soil_moisture', 'evapotranspiration_mm',
        'dtr', 'dewpoint_spread'
    ]

    for col in base_cols:
        for lag in [1, 2, 3]:
            df[f'{col}_lag_{lag}'] = df[col].shift(lag)
        df[f'{col}_roll3_mean'] = df[col].shift(1).rolling(3).mean()
        df[f'{col}_roll7_std'] = df[col].shift(1).rolling(7).std()

    return df.dropna().reset_index(drop=True)

df_historical = pd.read_csv(DATA_PATH)
df_historical_proc = preprocess_dataset(df_historical)

feature_cols = [c for c in df_historical_proc.columns if c not in ['date', 'rainfall_mm']]
X_train = df_historical_proc[feature_cols]
y_train = df_historical_proc['rainfall_mm']

# ==============================================================================
# STEP 2: CUSTOM ASYMMETRIC LOSS FUNCTION FOR EXTREME RAIN
# ==============================================================================
# Penalize under-predicting rainfall > 20mm much harder than over-predicting
def asymmetric_heavy_rain_loss(y_true, y_pred):
    residual = y_true - y_pred
    grad = np.where((y_true > 20.0) & (residual > 0), -4.0 * residual, -1.0 * residual)
    hess = np.where((y_true > 20.0) & (residual > 0), 4.0, 1.0)
    return grad, hess

print("[2/5] Training Asymmetric Hybrid Pipeline...")

# Stage 1: Rain Classifier
y_train_rain_binary = (y_train > 0.1).astype(int)
stage1_clf = LGBMClassifier(n_estimators=300, learning_rate=0.03, class_weight='balanced', random_state=42, verbose=-1)
stage1_clf.fit(X_train, y_train_rain_binary)

# Stage 2: Asymmetric Heavy Rain Regressor
rain_mask = y_train > 0.1
X_train_rainy = X_train[rain_mask]
y_train_rainy = y_train[rain_mask]

stage2_asym_reg = LGBMRegressor(n_estimators=400, learning_rate=0.03, random_state=42, verbose=-1)
stage2_asym_reg.set_params(objective=asymmetric_heavy_rain_loss)
stage2_asym_reg.fit(X_train_rainy, y_train_rainy)

# Save Pipeline
asym_bundle = {
    "stage1_clf": stage1_clf,
    "stage2_asym_reg": stage2_asym_reg,
    "threshold": 0.45
}
joblib.dump(asym_bundle, os.path.join(MODEL_DIR, "asymmetric_hybrid_pipeline.pkl"))

# ==============================================================================
# STEP 3: FETCH & PREPROCESS JULY 2026 TEST DATA
# ==============================================================================
print("[3/5] Extracting July 2026 data from Open-Meteo API...")
INDORE_LAT, INDORE_LON = 22.7196, 75.8577
api_url = "https://archive-api.open-meteo.com/v1/archive"
params = {
    "latitude": INDORE_LAT, "longitude": INDORE_LON,
    "start_date": "2026-06-15", "end_date": "2026-08-11",
    "hourly": [
        "temperature_2m", "relative_humidity_2m", "dew_point_2m",
        "surface_pressure", "shortwave_radiation", "wind_speed_10m",
        "soil_moisture_0_to_7cm", "et0_fao_evapotranspiration", "precipitation"
    ],
    "timezone": "Asia/Kolkata"
}

res = requests.get(api_url, params=params).json()['hourly']
df_h = pd.DataFrame(res)
df_h['time'] = pd.to_datetime(df_h['time'])
df_h['date'] = df_h['time'].dt.strftime('%d-%m-%Y')

df_d = df_h.groupby('date').agg(
    tmax_degC=('temperature_2m', 'max'), tmin_degC=('temperature_2m', 'min'),
    humidity_pct=('relative_humidity_2m', 'mean'),
    radiation_wm2=('shortwave_radiation', lambda x: x.mean() * 0.27778),
    wind_speed_ms=('wind_speed_10m', lambda x: x.mean() / 3.6),
    dewpoint_degC=('dew_point_2m', 'mean'), surface_pressure_hpa=('surface_pressure', 'mean'),
    soil_moisture=('soil_moisture_0_to_7cm', 'mean'), evapotranspiration_mm=('et0_fao_evapotranspiration', 'sum'),
    rainfall_mm=('precipitation', 'sum')
).reset_index()

df_d['date'] = pd.to_datetime(df_d['date'], format='%d-%m-%Y')
df_d = df_d.sort_values('date').reset_index(drop=True)

df_july = preprocess_dataset(df_d)
july_mask = (df_july['date'] >= '2026-07-01') & (df_july['date'] <= '2026-07-31')
df_july_test = df_july[july_mask].copy().reset_index(drop=True)

# ==============================================================================
# STEP 4: INFERENCE & METRIC EVALUATION
# ==============================================================================
print("[4/5] Running Inference on July 2026...")
X_july = df_july_test[feature_cols]
y_july_actual = df_july_test['rainfall_mm'].values

probs = stage1_clf.predict_proba(X_july)[:, 1]
raw_preds = stage2_asym_reg.predict(X_july)

final_preds = np.where(probs >= 0.45, raw_preds, 0.0)
final_preds = np.maximum(0.0, final_preds)
df_july_test['asym_predicted_mm'] = np.round(final_preds, 2)

def compute_metrics(y_true, y_pred):
    actual_rain = (y_true > 0.1).astype(int)
    pred_rain = (y_pred > 0.1).astype(int)
    cm = confusion_matrix(actual_rain, pred_rain, labels=[0, 1])
    tn, fp, fn, tp = cm.ravel()
    pod = tp / (tp + fn) if (tp + fn) > 0 else 0.0
    far = fp / (tp + fp) if (tp + fp) > 0 else 0.0
    csi = tp / (tp + fp + fn) if (tp + fp + fn) > 0 else 0.0

    actual_ext = (y_true > 30.0).astype(int)
    pred_ext = (y_pred > 30.0).astype(int)
    cm_ext = confusion_matrix(actual_ext, pred_ext, labels=[0, 1])
    e_tn, e_fp, e_fn, e_tp = cm_ext.ravel()
    ext_csi = e_tp / (e_tp + e_fp + e_fn) if (e_tp + e_fp + e_fn) > 0 else 0.0

    rmse = np.sqrt(mean_squared_error(y_true, y_pred))
    mae = mean_absolute_error(y_true, y_pred)
    return {
        "POD": round(pod, 4), "FAR": round(far, 4), "CSI": round(csi, 4),
        "Extreme_CSI (>30mm)": round(ext_csi, 4),
        "RMSE (mm)": round(rmse, 4), "MAE (mm)": round(mae, 4)
    }

metrics = compute_metrics(y_july_actual, final_preds)

print("\n==========================================================================================")
print("               JULY 2026 ASYMMETRIC LOSS HYBRID PERFORMANCE                               ")
print("==========================================================================================")
for k, v in metrics.items():
    print(f"{k}: {v}")
print("==========================================================================================")

# ==============================================================================
# STEP 5: VISUALIZATION
# ==============================================================================
print("[5/5] Generating Plot...")
plt.figure(figsize=(14, 6))
plt.plot(df_july_test['date'], df_july_test['rainfall_mm'], label='Actual Rainfall (mm)', color='#1f77b4', linewidth=2.5, marker='o')
plt.plot(df_july_test['date'], df_july_test['asym_predicted_mm'], label='Asymmetric Hybrid Prediction (mm)', color='#ff7f0e', linestyle='--', linewidth=2, marker='s')
plt.axhline(y=30.0, color='red', linestyle=':', label='Extreme Event Threshold (30mm)')
plt.title('Indore July 2026: Actual vs Asymmetric Loss Hybrid Model', fontsize=14, fontweight='bold')
plt.xlabel('Date', fontsize=12)
plt.ylabel('Rainfall Amount (mm)', fontsize=12)
plt.xticks(df_july_test['date'], df_july_test['date'].dt.strftime('%d-%b'), rotation=45)
plt.grid(True, alpha=0.3)
plt.legend(fontsize=11)
plt.tight_layout()
plt.savefig(os.path.join(JULY_DATA_DIR, "July_2026_Asymmetric_Hybrid.png"), dpi=300)
plt.show()

Block 1: Setup & Universal Data Extractor
Run this once to establish your workspace and fetch data for any location.

In [6]:
# ==============================================================================
# BLOCK 1: WORKSPACE SETUP & UNIVERSAL DATA EXTRACTOR
# ==============================================================================
import os
import json
import joblib
import requests
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from datetime import datetime
from google.colab import drive

# 1. Mount Drive & Setup App Directories
drive.mount('/content/drive', force_remount=False)

BASE_DIR = '/content/drive/MyDrive/BTP-COLLAB'
DATA_DIR = os.path.join(BASE_DIR, 'data')
MODEL_DIR = os.path.join(BASE_DIR, 'Models_new')
APP_DIR = os.path.join(BASE_DIR, 'App') # Target folder for Vercel/GitHub deployment

for directory in [DATA_DIR, MODEL_DIR, APP_DIR]:
    os.makedirs(directory, exist_ok=True)

print(f"✅ Workspace ready. Dashboard will be saved to: {APP_DIR}")

# 2. Universal Data Extraction Function
def fetch_historical_weather(lat, lon, start_date, end_date, output_filename):
    """
    Fetches daily ERA5 weather data for ANY given coordinates and saves to CSV.
    """
    print(f"🌍 Fetching data for Lat: {lat}, Lon: {lon} from {start_date} to {end_date}...")
    url = "https://archive-api.open-meteo.com/v1/archive"
    params = {
        "latitude": lat, "longitude": lon,
        "start_date": start_date, "end_date": end_date,
        "daily": [
            "temperature_2m_max", "temperature_2m_min", "precipitation_sum",
            "relative_humidity_2m_mean", "shortwave_radiation_sum",
            "wind_speed_10m_max", "dew_point_2m_mean", "surface_pressure_mean",
            "soil_moisture_0_to_7cm_mean", "et0_fao_evapotranspiration"
        ],
        "timezone": "auto"
    }

    response = requests.get(url, params=params)
    if response.status_code != 200:
        raise Exception(f"API Error {response.status_code}: {response.text}")

    df = pd.DataFrame(response.json()["daily"])
    df['date'] = pd.to_datetime(df['time']).dt.strftime('%d-%m-%Y')
    df['radiation_wm2'] = df['shortwave_radiation_sum'] * 11.574

    df.rename(columns={
        'temperature_2m_max': 'tmax_degC', 'temperature_2m_min': 'tmin_degC',
        'precipitation_sum': 'rainfall_mm', 'relative_humidity_2m_mean': 'humidity_pct',
        'wind_speed_10m_max': 'wind_speed_ms', 'dew_point_2m_mean': 'dewpoint_degC',
        'surface_pressure_mean': 'surface_pressure_hpa', 'soil_moisture_0_to_7cm_mean': 'soil_moisture',
        'et0_fao_evapotranspiration': 'evapotranspiration_mm'
    }, inplace=True)

    cols = ['date', 'tmax_degC', 'tmin_degC', 'rainfall_mm', 'humidity_pct',
            'radiation_wm2', 'wind_speed_ms', 'dewpoint_degC', 'surface_pressure_hpa',
            'soil_moisture', 'evapotranspiration_mm']
    df = df[cols].bfill().ffill()
    df['rainfall_mm'] = np.maximum(0.0, df['rainfall_mm'])

    output_path = os.path.join(DATA_DIR, output_filename)
    df.to_csv(output_path, index=False)
    print(f"✅ Data saved to {output_path} ({len(df)} records)")
    return output_path

# Example Usage: Extracting Indore Data
indore_csv = fetch_historical_weather(22.7196, 75.8577, "2026-07-01", "2026-08-11", "indore-rainfall-data-test.csv")

Block 2: Dynamic Feature Engineering
This block takes any CSV generated above and prepares it for machine learning.

In [7]:
# ==============================================================================
# BLOCK 2: DYNAMIC PREPROCESSING ENGINE
# ==============================================================================
def preprocess_and_split(filepath, train_ratio=0.8):
    """
    Loads CSV, applies temporal/physical feature engineering, and splits into train/test.
    """
    print(f"⚙️ Processing dataset: {filepath}")
    df = pd.read_csv(filepath)
    df['date'] = pd.to_datetime(df['date'], format='%d-%m-%Y')
    df = df.sort_values('date').reset_index(drop=True)

    # Temporal Features
    df['day_of_year'] = df['date'].dt.dayofyear
    df['sin_day'] = np.sin(2 * np.pi * df['day_of_year'] / 365.25)
    df['cos_day'] = np.cos(2 * np.pi * df['day_of_year'] / 365.25)

    # Physical Atmospheric Features
    df['dtr'] = df['tmax_degC'] - df['tmin_degC']
    df['temp_humidity_idx'] = df['tmax_degC'] * df['humidity_pct']
    df['dewpoint_spread'] = df['tmax_degC'] - df['dewpoint_degC']
    df['pressure_drop_1d'] = df['surface_pressure_hpa'].shift(1) - df['surface_pressure_hpa']

    # Rolling Windows & Lags
    base_cols = ['tmax_degC', 'tmin_degC', 'humidity_pct', 'radiation_wm2', 'wind_speed_ms',
                 'dewpoint_degC', 'surface_pressure_hpa', 'soil_moisture', 'evapotranspiration_mm',
                 'dtr', 'dewpoint_spread']

    for col in base_cols:
        for lag in [1, 2, 3]:
            df[f'{col}_lag_{lag}'] = df[col].shift(lag)
        df[f'{col}_roll3_mean'] = df[col].shift(1).rolling(3).mean()
        df[f'{col}_roll7_std'] = df[col].shift(1).rolling(7).std()

    df_clean = df.dropna().reset_index(drop=True)

    # Train-Test Split
    split_idx = int(len(df_clean) * train_ratio)
    feature_cols = [c for c in df_clean.columns if c not in ['date', 'rainfall_mm']]

    X_train, y_train = df_clean.iloc[:split_idx][feature_cols], df_clean.iloc[:split_idx]['rainfall_mm']
    X_test, y_test = df_clean.iloc[split_idx:][feature_cols], df_clean.iloc[split_idx:]['rainfall_mm']

    print(f"✅ Prepared {len(X_train)} train samples and {len(X_test)} test samples. Features: {len(feature_cols)}")
    return X_train, y_train, X_test, y_test, df_clean

X_train, y_train, X_test, y_test, full_df = preprocess_and_split(indore_csv)

Block 3: Model Training & Dashboard History Tracker
Here, we train the models, calculate metrics, and build a dictionary (history_tracker) that tracks all progress.

In [8]:
# ==============================================================================
# BLOCK 3: MODEL TRAINING & METRICS TRACKER
# ==============================================================================
from sklearn.metrics import mean_squared_error, mean_absolute_error, confusion_matrix
from lightgbm import LGBMClassifier, LGBMRegressor

# Initialize Global Tracker for the Dashboard
history_tracker = {
    "project_name": "Indore Rainfall Prediction Model",
    "last_updated": datetime.now().strftime("%Y-%m-%d %H:%M:%S"),
    "models": {}
}

def compute_metrics(y_true, y_pred):
    # Standard Rain (>0.1mm)
    cm = confusion_matrix((y_true > 0.1).astype(int), (y_pred > 0.1).astype(int), labels=[0, 1])
    tp = cm[1,1]; fp = cm[0,1]; fn = cm[1,0]
    csi = tp / (tp + fp + fn) if (tp + fp + fn) > 0 else 0.0

    # Extreme Rain (>30mm)
    cm_ext = confusion_matrix((y_true > 30.0).astype(int), (y_pred > 30.0).astype(int), labels=[0, 1])
    e_tp = cm_ext[1,1]; e_fp = cm_ext[0,1]; e_fn = cm_ext[1,0]
    ext_csi = e_tp / (e_tp + e_fp + e_fn) if (e_tp + e_fp + e_fn) > 0 else 0.0

    return {
        "RMSE": round(np.sqrt(mean_squared_error(y_true, y_pred)), 4),
        "MAE": round(mean_absolute_error(y_true, y_pred), 4),
        "CSI": round(csi, 4),
        "Extreme_CSI": round(ext_csi, 4)
    }

def log_model_to_tracker(model_name, description, y_true, y_pred):
    metrics = compute_metrics(y_true, y_pred)
    history_tracker["models"][model_name] = {
        "description": description,
        "metrics": metrics
    }
    print(f"📊 Logged {model_name} -> RMSE: {metrics['RMSE']}, Ext_CSI: {metrics['Extreme_CSI']}")

# ------------------------------------------------------------------------------
# Train Best Model: Tuned Asymmetric Hybrid
# ------------------------------------------------------------------------------
print("\n🚀 Training Tuned Asymmetric Hybrid (Current Best Method)...")

# Stage 1: Classifier
y_train_binary = (y_train > 0.1).astype(int)
stage1_clf = LGBMClassifier(n_estimators=300, learning_rate=0.03, class_weight='balanced', random_state=42, verbose=-1)
stage1_clf.fit(X_train, y_train_binary)

# Stage 2: Asymmetric Regressor (Penalty > 25mm)
def moderated_asymmetric_loss(y_true, y_pred):
    residual = y_true - y_pred
    asym_weight = np.where((y_true > 25.0) & (residual > 0), 1.8, 1.0)
    return -asym_weight * residual, asym_weight

rain_mask = y_train > 0.1
stage2_reg = LGBMRegressor(n_estimators=350, learning_rate=0.03, random_state=42, verbose=-1)
stage2_reg.set_params(objective=moderated_asymmetric_loss)
stage2_reg.fit(X_train[rain_mask], y_train[rain_mask])

# Inference on Test Set
probs = stage1_clf.predict_proba(X_test)[:, 1]
raw_preds = stage2_reg.predict(X_test)
final_preds = np.maximum(0.0, np.where(probs >= 0.45, raw_preds, 0.0))

# Save Model & Log to Tracker
joblib.dump({"clf": stage1_clf, "reg": stage2_reg}, os.path.join(MODEL_DIR, "tuned_asym_hybrid.pkl"))
log_model_to_tracker("Tuned Asymmetric Hybrid", "1.8x Penalty on >25mm missed events", y_test, final_preds)

# You can easily add earlier models to the tracker here as well (e.g., CatBoost, Standard Hybrid) to populate the dashboard.

In [ ]:
!pip install ee pandas numpy matplotlib joblib datetime lightbgm sklearn

In [5]:


import ee
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import joblib
from datetime import datetime
from lightgbm import LGBMClassifier, LGBMRegressor
from sklearn.metrics import mean_squared_error, mean_absolute_error, confusion_matrix

# Authenticate and initialize Earth Engine
# Replace 'your-google-cloud-project-id' with your active GCP Project ID


ee.Authenticate(auth_mode='notebook')
ee.Initialize(project='raincast-ai')

In [ ]:
def fetch_gee_weather_data(lat, lon, start_date, end_date):
    """
    Extracts daily ERA5-Land climate data directly from GEE for a given location.
    """
    print(f"🌍 Querying Google Earth Engine for Lat: {lat}, Lon: {lon} ({start_date} to {end_date})...")

    point = ee.Geometry.Point([lon, lat])

    # Load ERA5-Land Daily Aggregated ImageCollection
    era5 = (
        ee.ImageCollection("ECMWF/ERA5_LAND/DAILY_AGGR")
        .filterDate(start_date, end_date)
        .select([
            'temperature_2m_max',
            'temperature_2m_min',
            'total_precipitation_sum',
            'dewpoint_temperature_2m',
            'surface_pressure',
            'surface_solar_radiation_downwards_sum',
            'volumetric_soil_water_layer_1'
        ])
    )

    # Extract timeseries at point geometry
    raw_info = era5.getRegion(point, scale=10000).getInfo()

    # Format GEE nested list into Pandas DataFrame
    headers = raw_info[0]
    data = raw_info[1:]
    df = pd.DataFrame(data, columns=headers)

    # Datetime conversion
    df['date'] = pd.to_datetime(df['time'], unit='ms')
    df = df.sort_values('date').reset_index(drop=True)

    # Convert GEE units to standard meteorological units
    df['tmax_degC'] = df['temperature_2m_max'] - 273.15
    df['tmin_degC'] = df['temperature_2m_min'] - 273.15
    df['dewpoint_degC'] = df['dewpoint_temperature_2m'] - 273.15
    df['rainfall_mm'] = np.maximum(0.0, df['total_precipitation_sum'] * 1000.0) # m -> mm
    df['surface_pressure_hpa'] = df['surface_pressure'] / 100.0 # Pa -> hPa
    df['soil_moisture'] = df['volumetric_soil_water_layer_1']
    df['radiation_wm2'] = df['surface_solar_radiation_downwards_sum'] / 86400.0 # J/m²/day -> W/m²

    # Approximate mean daily temperature & Calculate Relative Humidity (%)
    tmean = (df['tmax_degC'] + df['tmin_degC']) / 2.0
    es = 6.112 * np.exp((17.67 * tmean) / (tmean + 243.5))
    e = 6.112 * np.exp((17.67 * df['dewpoint_degC']) / (df['dewpoint_degC'] + 243.5))
    df['humidity_pct'] = np.clip((e / es) * 100.0, 0, 100)

    # Estimate wind speed surrogate & dummy evapotranspiration for feature compatibility
    df['wind_speed_ms'] = 2.5
    df['evapotranspiration_mm'] = np.maximum(0.1, (df['tmax_degC'] - df['tmin_degC']) * 0.2)

    cols = [
        'date', 'tmax_degC', 'tmin_degC', 'rainfall_mm', 'humidity_pct',
        'radiation_wm2', 'wind_speed_ms', 'dewpoint_degC',
        'surface_pressure_hpa', 'soil_moisture', 'evapotranspiration_mm'
    ]

    df_out = df[cols].bfill().ffill()
    print(f"✅ Downloaded {len(df_out)} daily records from GEE.")
    return df_out

In [ ]:
def preprocess_gee_dataset(df):
    """
    Applies temporal encodings, atmospheric physics features, lags, and rolling windows.
    """
    df_proc = df.copy()

    # Temporal cyclical features
    df_proc['day_of_year'] = df_proc['date'].dt.dayofyear
    df_proc['sin_day'] = np.sin(2 * np.pi * df_proc['day_of_year'] / 365.25)
    df_proc['cos_day'] = np.cos(2 * np.pi * df_proc['day_of_year'] / 365.25)

    # Atmospheric Physics Features
    df_proc['dtr'] = df_proc['tmax_degC'] - df_proc['tmin_degC']
    df_proc['temp_humidity_idx'] = df_proc['tmax_degC'] * df_proc['humidity_pct']
    df_proc['dewpoint_spread'] = df_proc['tmax_degC'] - df_proc['dewpoint_degC']
    df_proc['pressure_drop_1d'] = df_proc['surface_pressure_hpa'].shift(1) - df_proc['surface_pressure_hpa']

    base_cols = [
        'tmax_degC', 'tmin_degC', 'humidity_pct', 'radiation_wm2', 'wind_speed_ms',
        'dewpoint_degC', 'surface_pressure_hpa', 'soil_moisture', 'evapotranspiration_mm',
        'dtr', 'dewpoint_spread'
    ]

    # Lagged and Rolling Window Features
    for col in base_cols:
        for lag in [1, 2, 3]:
            df_proc[f'{col}_lag_{lag}'] = df_proc[col].shift(lag)
        df_proc[f'{col}_roll3_mean'] = df_proc[col].shift(1).rolling(3).mean()
        df_proc[f'{col}_roll7_std'] = df_proc[col].shift(1).rolling(7).std()

    return df_proc.dropna().reset_index(drop=True)

In [ ]:
# 1. Coordinate Setup (e.g., Indore)
INDORE_LAT, INDORE_LON = 22.7196, 75.8577

# 2. Extract Training & Test Datasets Directly from GEE
df_train_raw = fetch_gee_weather_data(INDORE_LAT, INDORE_LON, "2015-01-01", "2025-12-31")
df_test_raw  = fetch_gee_weather_data(INDORE_LAT, INDORE_LON, "2026-06-15", "2026-07-31")

# 3. Preprocess
df_train = preprocess_gee_dataset(df_train_raw)
df_test  = preprocess_gee_dataset(df_test_raw)

# Filter test set specifically for July 2026
july_mask = (df_test['date'] >= '2026-07-01') & (df_test['date'] <= '2026-07-31')
df_test_july = df_test[july_mask].copy().reset_index(drop=True)

feature_cols = [c for c in df_train.columns if c not in ['date', 'rainfall_mm']]

X_train, y_train = df_train[feature_cols], df_train['rainfall_mm']
X_test, y_test   = df_test_july[feature_cols], df_test_july['rainfall_mm'].values

# 4. Train Stage 1 Classifier (Rain / No-Rain)
y_train_binary = (y_train > 0.1).astype(int)
stage1_clf = LGBMClassifier(n_estimators=300, learning_rate=0.03, class_weight='balanced', random_state=42, verbose=-1)
stage1_clf.fit(X_train, y_train_binary)

# 5. Train Stage 2 Asymmetric Loss Regressor
def asymmetric_heavy_rain_loss(y_true, y_pred):
    residual = y_true - y_pred
    grad = np.where((y_true > 20.0) & (residual > 0), -4.0 * residual, -1.0 * residual)
    hess = np.where((y_true > 20.0) & (residual > 0), 4.0, 1.0)
    return grad, hess

rain_mask = y_train > 0.1
stage2_asym_reg = LGBMRegressor(n_estimators=400, learning_rate=0.03, random_state=42, verbose=-1)
stage2_asym_reg.set_params(objective=asymmetric_heavy_rain_loss)
stage2_asym_reg.fit(X_train[rain_mask], y_train[rain_mask])

# 6. Forecasting / Inference
probs = stage1_clf.predict_proba(X_test)[:, 1]
raw_preds = stage2_asym_reg.predict(X_test)
final_preds = np.where(probs >= 0.45, raw_preds, 0.0)
final_preds = np.maximum(0.0, final_preds)

df_test_july['predicted_mm'] = np.round(final_preds, 2)

# 7. Compute Performance Metrics
def evaluate_forecast(y_true, y_pred):
    actual_rain = (y_true > 0.1).astype(int)
    pred_rain = (y_pred > 0.1).astype(int)
    cm = confusion_matrix(actual_rain, pred_rain, labels=[0, 1])
    tn, fp, fn, tp = cm.ravel()

    pod = tp / (tp + fn) if (tp + fn) > 0 else 0.0
    far = fp / (tp + fp) if (tp + fp) > 0 else 0.0
    csi = tp / (tp + fp + fn) if (tp + fp + fn) > 0 else 0.0

    actual_ext = (y_true > 30.0).astype(int)
    pred_ext = (y_pred > 30.0).astype(int)
    cm_ext = confusion_matrix(actual_ext, pred_ext, labels=[0, 1])
    e_tn, e_fp, e_fn, e_tp = cm_ext.ravel()
    ext_csi = e_tp / (e_tp + e_fp + e_fn) if (e_tp + e_fp + e_fn) > 0 else 0.0

    rmse = np.sqrt(mean_squared_error(y_true, y_pred))
    mae = mean_absolute_error(y_true, y_pred)

    return {"POD": round(pod, 4), "FAR": round(far, 4), "CSI": round(csi, 4),
            "Extreme_CSI (>30mm)": round(ext_csi, 4), "RMSE (mm)": round(rmse, 4), "MAE (mm)": round(mae, 4)}

metrics = evaluate_forecast(y_test, final_preds)
print("\n================ GEE MODEL INFERENCE RESULTS ================")
for k, v in metrics.items():
    print(f"{k}: {v}")

# 8. Visualization
plt.figure(figsize=(14, 5))
plt.plot(df_test_july['date'], df_test_july['rainfall_mm'], label='Actual Rainfall (GEE ERA5)', color='#1f77b4', linewidth=2.5, marker='o')
plt.plot(df_test_july['date'], df_test_july['predicted_mm'], label='Asymmetric Hybrid Prediction', color='#ff7f0e', linestyle='--', linewidth=2, marker='s')
plt.axhline(y=30.0, color='red', linestyle=':', label='Extreme Event Threshold (30mm)')
plt.title('July 2026: GEE Direct Data Training & Forecasting', fontsize=13, fontweight='bold')
plt.xlabel('Date'); plt.ylabel('Rainfall (mm)')
plt.xticks(df_test_july['date'], df_test_july['date'].dt.strftime('%d-%b'), rotation=45)
plt.grid(True, alpha=0.3); plt.legend()
plt.tight_layout(); plt.show()

In [ ]:
import os
import joblib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.metrics import mean_squared_error, mean_absolute_error, confusion_matrix

# ==============================================================================
# 1. PATHS & DATA PREPARATION
# ==============================================================================
BASE_DIR = '/content/drive/MyDrive/BTP_COLLAB'
MODEL_DIR = os.path.join(BASE_DIR, 'Models_new')
DATA_PATH = os.path.join(BASE_DIR, 'data', 'indore-rainfall-data.csv')

# Preprocessing helper (reusing feature engineering pipeline)
def preprocess_dataset(filepath):
    df = pd.read_csv(filepath)
    df['date'] = pd.to_datetime(df['date'], format='%d-%m-%Y', errors='coerce').fillna(
        pd.to_datetime(df['date'], errors='coerce')
    )
    df = df.sort_values('date').reset_index(drop=True)

    # Temporal & physical features
    df['day_of_year'] = df['date'].dt.dayofyear
    df['sin_day'] = np.sin(2 * np.pi * df['day_of_year'] / 365.25)
    df['cos_day'] = np.cos(2 * np.pi * df['day_of_year'] / 365.25)
    df['dtr'] = df['tmax_degC'] - df['tmin_degC']
    df['temp_humidity_idx'] = df['tmax_degC'] * df['humidity_pct']
    df['dewpoint_spread'] = df['tmax_degC'] - df['dewpoint_degC']
    df['pressure_drop_1d'] = df['surface_pressure_hpa'].shift(1) - df['surface_pressure_hpa']

    base_cols = [
        'tmax_degC', 'tmin_degC', 'humidity_pct', 'radiation_wm2', 'wind_speed_ms',
        'dewpoint_degC', 'surface_pressure_hpa', 'soil_moisture', 'evapotranspiration_mm',
        'dtr', 'dewpoint_spread'
    ]

    for col in base_cols:
        for lag in [1, 2, 3]:
            df[f'{col}_lag_{lag}'] = df[col].shift(lag)
        df[f'{col}_roll3_mean'] = df[col].shift(1).rolling(3).mean()
        df[f'{col}_roll7_std'] = df[col].shift(1).rolling(7).std()

    return df.dropna().reset_index(drop=True)

# Load dataset and extract July test records (e.g., July 2025 or latest July in dataset)
df_all = preprocess_dataset(DATA_PATH)
df_july = df_all[df_all['date'].dt.month == 7].copy().reset_index(drop=True)

# Take the most recent July available in your dataset for evaluation
latest_year = df_july['date'].dt.year.max()
df_july_test = df_july[df_july['date'].dt.year == latest_year].copy().reset_index(drop=True)

feature_cols = [c for c in df_july_test.columns if c not in ['date', 'rainfall_mm']]
X_july = df_july_test[feature_cols]
y_july = df_july_test['rainfall_mm'].values

print(f"📊 Evaluating models on July {latest_year} ({len(df_july_test)} daily records)...")

# ==============================================================================
# 2. METRICS EVALUATOR FUNCTION
# ==============================================================================
def compute_metrics(y_true, y_pred):
    # Categorical Rain / No-Rain Metrics
    actual_rain = (y_true > 0.1).astype(int)
    pred_rain = (y_pred > 0.1).astype(int)
    cm = confusion_matrix(actual_rain, pred_rain, labels=[0, 1])
    tn, fp, fn, tp = cm.ravel()

    pod = tp / (tp + fn) if (tp + fn) > 0 else 0.0
    far = fp / (tp + fp) if (tp + fp) > 0 else 0.0
    csi = tp / (tp + fp + fn) if (tp + fp + fn) > 0 else 0.0

    # Extreme Rain (>30mm)
    actual_ext = (y_true > 30.0).astype(int)
    pred_ext = (y_pred > 30.0).astype(int)
    cm_ext = confusion_matrix(actual_ext, pred_ext, labels=[0, 1])
    e_tn, e_fp, e_fn, e_tp = cm_ext.ravel()
    ext_csi = e_tp / (e_tp + e_fp + e_fn) if (e_tp + e_fp + e_fn) > 0 else 0.0

    rmse = np.sqrt(mean_squared_error(y_true, y_pred))
    mae = mean_absolute_error(y_true, y_pred)

    return {
        "RMSE (mm)": round(rmse, 4),
        "MAE (mm)": round(mae, 4),
        "CSI": round(csi, 4),
        "POD": round(pod, 4),
        "FAR": round(far, 4),
        "Extreme CSI (>30mm)": round(ext_csi, 4)
    }

# ==============================================================================
# 3. DYNAMIC MODEL SCANNER & INFERENCE ENGINE
# ==============================================================================
model_files = [f for f in os.listdir(MODEL_DIR) if f.endswith(('.pkl', '.joblib'))]

if not model_files:
    print(f"⚠️ No .pkl or .joblib models found in {MODEL_DIR}")
else:
    results = []
    predictions_dict = {"Date": df_july_test['date'], "Actual Rainfall": y_july}

    for file_name in model_files:
        model_path = os.path.join(MODEL_DIR, file_name)
        model_name = file_name.replace('.pkl', '').replace('.joblib', '')

        try:
            model_obj = joblib.load(model_path)

            # Case 1: Two-Stage Hybrid Dictionary
            if isinstance(model_obj, dict):
                clf = model_obj.get("clf") or model_obj.get("stage1_clf")
                reg = model_obj.get("reg") or model_obj.get("stage2_asym_reg") or model_obj.get("stage2_reg")
                thresh = model_obj.get("threshold", 0.45)

                probs = clf.predict_proba(X_july)[:, 1]
                raw_preds = reg.predict(X_july)
                preds = np.where(probs >= thresh, raw_preds, 0.0)
                preds = np.maximum(0.0, preds)

            # Case 2: Single Regressor / Model Object
            else:
                preds = np.maximum(0.0, model_obj.predict(X_july))

            # Store predictions and calculate metrics
            predictions_dict[model_name] = preds
            metrics = compute_metrics(y_july, preds)
            metrics["Model Name"] = model_name
            results.append(metrics)
            print(f"✅ Executed: {model_name}")

        except Exception as e:
            print(f"❌ Failed to evaluate {file_name}: {e}")

    # ==============================================================================
    # 4. COMPARISON TABLE
    # ==============================================================================
    df_results = pd.DataFrame(results)
    cols_order = ["Model Name", "RMSE (mm)", "MAE (mm)", "CSI", "Extreme CSI (>30mm)", "POD", "FAR"]
    df_results = df_results[cols_order].sort_values(by="Extreme CSI (>30mm)", ascending=False).reset_index(drop=True)

    print("\n==========================================================================================")
    print(f"                           JULY {latest_year} MODEL COMPARISON TABLE                             ")
    print("==========================================================================================")
    print(df_results.to_string(index=False))
    print("==========================================================================================\n")

    # ==============================================================================
    # 5. VISUAL COMPARISON PLOT
    # ==============================================================================
    plt.figure(figsize=(15, 7))
    plt.plot(df_july_test['date'], y_july, label='Actual Rainfall (mm)', color='black', linewidth=3, marker='o')

    colors = ['#ff7f0e', '#2ca02c', '#d62728', '#9467bd', '#8c564b', '#e377c2']
    for idx, name in enumerate(df_results["Model Name"]):
        plt.plot(
            df_july_test['date'],
            predictions_dict[name],
            label=f'Predicted: {name}',
            linestyle='--',
            linewidth=2,
            color=colors[idx % len(colors)],
            marker='s'
        )

    plt.axhline(y=30.0, color='red', linestyle=':', label='Extreme Event Threshold (30mm)')
    plt.title(f'Indore July {latest_year}: Model Performance Comparison', fontsize=14, fontweight='bold')
    plt.xlabel('Date', fontsize=12)
    plt.ylabel('Rainfall Amount (mm)', fontsize=12)
    plt.xticks(df_july_test['date'], df_july_test['date'].dt.strftime('%d-%b'), rotation=45)
    plt.grid(True, alpha=0.3)
    plt.legend(fontsize=10, loc='upper right')
    plt.tight_layout()
    plt.show()

SRNAG+

In [ ]:
!pip install earthengine-api CatBoost



In [ ]:
import os
import requests
import joblib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_squared_error, mean_absolute_error, confusion_matrix
from google.colab import drive

# ==============================================================================
# STEP 1: INITIALIZATION & DATA ACQUISITION (1951–2025 Open-Meteo ERA5)
# ==============================================================================
print("[1/6] Mounting Google Drive & initializing workspace...")
drive.mount('/content/drive', force_remount=False)

BASE_DIR = '/content/drive/MyDrive/BTP-COLLAB'
DATA_DIR = os.path.join(BASE_DIR, 'data')
MODEL_DIR = os.path.join(BASE_DIR, 'Models_new')
os.makedirs(DATA_DIR, exist_ok=True)
os.makedirs(MODEL_DIR, exist_ok=True)

CSV_PATH = os.path.join(DATA_DIR, 'indore-rainfall-1951-2025.csv')
INDORE_LAT, INDORE_LON = 22.7196, 75.8577

if not os.path.exists(CSV_PATH):
    print("🌍 Fetching 75 Years (1951-2025) Data from Open-Meteo API...")
    url = "https://archive-api.open-meteo.com/v1/archive"
    params = {
        "latitude": INDORE_LAT,
        "longitude": INDORE_LON,
        "start_date": "1951-01-01",
        "end_date": "2025-12-31",
        "daily": [
            "temperature_2m_max", "temperature_2m_min", "precipitation_sum",
            "relative_humidity_2m_mean", "shortwave_radiation_sum",
            "wind_speed_10m_max", "dew_point_2m_mean", "surface_pressure_mean",
            "soil_moisture_0_to_7cm_mean", "et0_fao_evapotranspiration"
        ],
        "timezone": "auto"
    }
    resp = requests.get(url, params=params)
    if resp.status_code != 200:
        raise RuntimeError(f"Open-Meteo Fetch Failed: {resp.status_code}")

    df_raw = pd.DataFrame(resp.json()["daily"])
    df_raw['date'] = pd.to_datetime(df_raw['time']).dt.strftime('%d-%m-%Y')
    df_raw['radiation_wm2'] = df_raw['shortwave_radiation_sum'] * 11.574
    df_raw.rename(columns={
        'temperature_2m_max': 'tmax_degC', 'temperature_2m_min': 'tmin_degC',
        'precipitation_sum': 'rainfall_mm', 'relative_humidity_2m_mean': 'humidity_pct',
        'wind_speed_10m_max': 'wind_speed_ms', 'dew_point_2m_mean': 'dewpoint_degC',
        'surface_pressure_mean': 'surface_pressure_hpa',
        'soil_moisture_0_to_7cm_mean': 'soil_moisture',
        'et0_fao_evapotranspiration': 'evapotranspiration_mm'
    }, inplace=True)

    cols = ['date', 'tmax_degC', 'tmin_degC', 'rainfall_mm', 'humidity_pct',
            'radiation_wm2', 'wind_speed_ms', 'dewpoint_degC', 'surface_pressure_hpa',
            'soil_moisture', 'evapotranspiration_mm']
    df_raw = df_raw[cols].bfill().ffill()
    df_raw['rainfall_mm'] = np.maximum(0.0, df_raw['rainfall_mm'])
    df_raw.to_csv(CSV_PATH, index=False)
    print(f"✅ Downloaded and saved {len(df_raw)} daily records to {CSV_PATH}")
else:
    print(f"✅ Found existing 75-year dataset at {CSV_PATH}")

# ==============================================================================
# STEP 2: FEATURE ENGINEERING & DATASET PREPARATION
# ==============================================================================
def preprocess_dataset(filepath):
    df = pd.read_csv(filepath)
    df['date'] = pd.to_datetime(df['date'], format='%d-%m-%Y', errors='coerce').fillna(
        pd.to_datetime(df['date'], errors='coerce')
    )
    df = df.sort_values('date').reset_index(drop=True)

    df['day_of_year'] = df['date'].dt.dayofyear
    df['sin_day'] = np.sin(2 * np.pi * df['day_of_year'] / 365.25)
    df['cos_day'] = np.cos(2 * np.pi * df['day_of_year'] / 365.25)
    df['dtr'] = df['tmax_degC'] - df['tmin_degC']
    df['temp_humidity_idx'] = df['tmax_degC'] * df['humidity_pct']
    df['dewpoint_spread'] = df['tmax_degC'] - df['dewpoint_degC']
    df['pressure_drop_1d'] = df['surface_pressure_hpa'].shift(1) - df['surface_pressure_hpa']

    base_cols = [
        'tmax_degC', 'tmin_degC', 'humidity_pct', 'radiation_wm2', 'wind_speed_ms',
        'dewpoint_degC', 'surface_pressure_hpa', 'soil_moisture', 'evapotranspiration_mm',
        'dtr', 'dewpoint_spread'
    ]

    for col in base_cols:
        for lag in [1, 2, 3]:
            df[f'{col}_lag_{lag}'] = df[col].shift(lag)
        df[f'{col}_roll3_mean'] = df[col].shift(1).rolling(3).mean()
        df[f'{col}_roll7_std'] = df[col].shift(1).rolling(7).std()

    return df.dropna().reset_index(drop=True)

print("[2/6] Processing features and building tensor matrices...")
full_df = preprocess_dataset(CSV_PATH)

feature_cols = [c for c in full_df.columns if c not in ['date', 'rainfall_mm']]
X_all = full_df[feature_cols].values
y_all = full_df['rainfall_mm'].values

# Split off last available July for evaluation testing
july_mask = (full_df['date'].dt.month == 7) & (full_df['date'].dt.year == full_df['date'].dt.year.max())
test_indices = full_df[july_mask].index

train_mask = ~full_df.index.isin(test_indices)
X_train_raw, y_train = X_all[train_mask], y_all[train_mask]
X_test_raw, y_test = X_all[july_mask], y_all[july_mask]
df_july_test = full_df[july_mask].copy().reset_index(drop=True)

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train_raw)
X_test_scaled = scaler.transform(X_test_raw)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

train_dataset = TensorDataset(torch.FloatTensor(X_train_scaled), torch.FloatTensor(y_train).unsqueeze(1))
train_loader = DataLoader(train_dataset, batch_size=128, shuffle=True)

# ==============================================================================
# STEP 3: MODEL 1 - HYBRID CONV1D-MLP MODEL
# ==============================================================================
print("[3/6] Building & Training Hybrid Conv1D-MLP Model...")

class HybridConv1DMLP(nn.Module):
    def __init__(self, input_dim):
        super(HybridConv1DMLP, self).__init__()
        # Conv1D feature extractor
        self.conv1 = nn.Conv1d(in_channels=1, out_channels=32, kernel_size=3, padding=1)
        self.bn1 = nn.BatchNorm1d(32)
        self.conv2 = nn.Conv1d(in_channels=32, out_channels=64, kernel_size=3, padding=1)
        self.bn2 = nn.BatchNorm1d(64)
        self.relu = nn.ReLU()

        # Dense MLP Classifier & Regressor head
        self.mlp = nn.Sequential(
            nn.Linear(64 * input_dim, 128),
            nn.ReLU(),
            nn.Dropout(0.2),
            nn.Linear(128, 64),
            nn.ReLU(),
            nn.Linear(64, 1)
        )

    def forward(self, x):
        # x shape: (batch, features) -> expand to (batch, 1, features)
        x = x.unsqueeze(1)
        x = self.relu(self.bn1(self.conv1(x)))
        x = self.relu(self.bn2(self.conv2(x)))
        x = x.view(x.size(0), -1)
        out = self.mlp(x)
        return torch.relu(out) # Precipitation non-negative

conv_mlp_model = HybridConv1DMLP(len(feature_cols)).to(device)
optimizer = optim.Adam(conv_mlp_model.parameters(), lr=0.001, weight_decay=1e-5)
criterion = nn.MSELoss()

conv_mlp_model.train()
for epoch in range(35):
    for bx, by in train_loader:
        bx, by = bx.to(device), by.to(device)
        optimizer.zero_grad()
        preds = conv_mlp_model(bx)
        loss = criterion(preds, by)
        loss.backward()
        optimizer.step()

# ==============================================================================
# STEP 4: MODEL 2 - 1D SUPER-RESOLUTION GAN (SRGAN)
# ==============================================================================
print("[4/6] Building & Training 1D-SRGAN (Generator & Discriminator)...")

class ResBlock1D(nn.Module):
    def __init__(self, channels):
        super(ResBlock1D, self).__init__()
        self.block = nn.Sequential(
            nn.Conv1d(channels, channels, kernel_size=3, padding=1),
            nn.BatchNorm1d(channels),
            nn.PReLU(),
            nn.Conv1d(channels, channels, kernel_size=3, padding=1),
            nn.BatchNorm1d(channels)
        )

    def forward(self, x):
        return x + self.block(x)

class SRGAN1D_Generator(nn.Module):
    def __init__(self, input_dim):
        super(SRGAN1D_Generator, self).__init__()
        self.init_conv = nn.Sequential(
            nn.Conv1d(1, 64, kernel_size=3, padding=1),
            nn.PReLU()
        )
        self.res_blocks = nn.Sequential(*[ResBlock1D(64) for _ in range(3)])
        self.mid_conv = nn.Sequential(
            nn.Conv1d(64, 64, kernel_size=3, padding=1),
            nn.BatchNorm1d(64)
        )
        self.head = nn.Sequential(
            nn.Linear(64 * input_dim, 128),
            nn.PReLU(),
            nn.Linear(128, 1)
        )

    def forward(self, x):
        x = x.unsqueeze(1)
        feat = self.init_conv(x)
        res = self.res_blocks(feat)
        out = feat + self.mid_conv(res)
        out = out.view(out.size(0), -1)
        return torch.relu(self.head(out))

class SRGAN1D_Discriminator(nn.Module):
    def __init__(self):
        super(SRGAN1D_Discriminator, self).__init__()
        self.net = nn.Sequential(
            nn.Linear(1, 32),
            nn.LeakyReLU(0.2),
            nn.Linear(32, 16),
            nn.LeakyReLU(0.2),
            nn.Linear(16, 1),
            nn.Sigmoid()
        )

    def forward(self, x):
        return self.net(x)

netG = SRGAN1D_Generator(len(feature_cols)).to(device)
netD = SRGAN1D_Discriminator().to(device)

optG = optim.Adam(netG.parameters(), lr=0.0003, betas=(0.5, 0.999))
optD = optim.Adam(netD.parameters(), lr=0.0001, betas=(0.5, 0.999))

l1_loss = nn.L1Loss()
bce_loss = nn.BCELoss()

netG.train()
netD.train()
for epoch in range(30):
    for bx, by in train_loader:
        bx, by = bx.to(device), by.to(device)
        batch_sz = bx.size(0)

        # Train Discriminator
        optD.zero_grad()
        fake_rain = netG(bx).detach()
        real_labels = torch.ones(batch_sz, 1).to(device)
        fake_labels = torch.zeros(batch_sz, 1).to(device)

        d_loss_real = bce_loss(netD(by), real_labels)
        d_loss_fake = bce_loss(netD(fake_rain), fake_labels)
        d_loss = (d_loss_real + d_loss_fake) / 2.0
        d_loss.backward()
        optD.step()

        # Train Generator (Content Loss + Adversarial Loss)
        optG.zero_grad()
        gen_rain = netG(bx)
        content_loss = l1_loss(gen_rain, by)
        adv_loss = bce_loss(netD(gen_rain), real_labels)
        g_loss = content_loss + 0.01 * adv_loss
        g_loss.backward()
        optG.step()

# ==============================================================================
# STEP 5: SAVE WRAPPER PIPELINES TO DRIVE
# ==============================================================================
print(f"[5/6] Saving PyTorch pipelines into {MODEL_DIR}...")

class PyTorchModelWrapper:
    def __init__(self, model, scaler):
        self.model = model.cpu().eval()
        self.scaler = scaler

    def predict(self, X):
        X_sc = self.scaler.transform(X)
        with torch.no_grad():
            t_X = torch.FloatTensor(X_sc)
            preds = self.model(t_X).numpy().flatten()
        return np.maximum(0.0, preds)

joblib.dump(PyTorchModelWrapper(conv_mlp_model, scaler), os.path.join(MODEL_DIR, "Hybrid_Conv1D_MLP.pkl"))
joblib.dump(PyTorchModelWrapper(netG, scaler), os.path.join(MODEL_DIR, "SRGAN_1D_Generator.pkl"))
print("✅ Saved 'Hybrid_Conv1D_MLP.pkl' and 'SRGAN_1D_Generator.pkl'")

# ==============================================================================
# STEP 6: BENCHMARK & COMPARE ALL MODELS IN MODEL_DIR
# ==============================================================================
print("\n[6/6] Executing multi-model benchmark evaluation on July test data...")

def compute_metrics(y_true, y_pred):
    actual_rain = (y_true > 0.1).astype(int)
    pred_rain = (y_pred > 0.1).astype(int)
    cm = confusion_matrix(actual_rain, pred_rain, labels=[0, 1])
    tn, fp, fn, tp = cm.ravel()

    pod = tp / (tp + fn) if (tp + fn) > 0 else 0.0
    far = fp / (tp + fp) if (tp + fp) > 0 else 0.0
    csi = tp / (tp + fp + fn) if (tp + fp + fn) > 0 else 0.0

    actual_ext = (y_true > 30.0).astype(int)
    pred_ext = (y_pred > 30.0).astype(int)
    cm_ext = confusion_matrix(actual_ext, pred_ext, labels=[0, 1])
    e_tn, e_fp, e_fn, e_tp = cm_ext.ravel()
    ext_csi = e_tp / (e_tp + e_fp + e_fn) if (e_tp + e_fp + e_fn) > 0 else 0.0

    rmse = np.sqrt(mean_squared_error(y_true, y_pred))
    mae = mean_absolute_error(y_true, y_pred)

    return {
        "RMSE (mm)": round(rmse, 4),
        "MAE (mm)": round(mae, 4),
        "CSI": round(csi, 4),
        "Extreme CSI (>30mm)": round(ext_csi, 4),
        "POD": round(pod, 4),
        "FAR": round(far, 4)
    }

model_files = [f for f in os.listdir(MODEL_DIR) if f.endswith(('.pkl', '.joblib'))]
results = []
predictions_dict = {"Date": df_july_test['date'], "Actual": y_test}

for file_name in model_files:
    model_path = os.path.join(MODEL_DIR, file_name)
    model_name = file_name.replace('.pkl', '').replace('.joblib', '')

    try:
        model_obj = joblib.load(model_path)

        # Dictionary Hybrid Pipeline
        if isinstance(model_obj, dict):
            clf = model_obj.get("clf") or model_obj.get("stage1_clf")
            reg = model_obj.get("reg") or model_obj.get("stage2_asym_reg") or model_obj.get("stage2_reg")
            thresh = model_obj.get("threshold", 0.45)
            probs = clf.predict_proba(X_test_raw)[:, 1]
            raw_preds = reg.predict(X_test_raw)
            preds = np.where(probs >= thresh, raw_preds, 0.0)
            preds = np.maximum(0.0, preds)
        # Object Wrapper or Scikit-Learn Model
        else:
            preds = np.maximum(0.0, model_obj.predict(X_test_raw))

        predictions_dict[model_name] = preds
        metrics = compute_metrics(y_test, preds)
        metrics["Model Name"] = model_name
        results.append(metrics)
    except Exception as e:
        print(f"⚠️ Error evaluating {file_name}: {e}")

df_results = pd.DataFrame(results)
cols_order = ["Model Name", "RMSE (mm)", "MAE (mm)", "CSI", "Extreme CSI (>30mm)", "POD", "FAR"]
df_results = df_results[cols_order].sort_values(by="Extreme CSI (>30mm)", ascending=False).reset_index(drop=True)

print("\n==========================================================================================")
print(f"                       JULY {df_july_test['date'].dt.year.max()} ALL-MODEL PERFORMANCE BENCHMARK                       ")
print("==========================================================================================")
print(df_results.to_string(index=False))
print("==========================================================================================\n")

# Visualization
plt.figure(figsize=(15, 6))
plt.plot(df_july_test['date'], y_test, label='Actual Rainfall (mm)', color='black', linewidth=3, marker='o')

colors = ['#1f77b4', '#ff7f0e', '#2ca02c', '#d62728', '#9467bd', '#8c564b', '#e377c2']
for idx, name in enumerate(df_results["Model Name"]):
    plt.plot(
        df_july_test['date'],
        predictions_dict[name],
        label=f'{name}',
        linestyle='--',
        linewidth=2,
        color=colors[idx % len(colors)],
        marker='s'
    )

plt.axhline(y=30.0, color='red', linestyle=':', label='Extreme Event Threshold (30mm)')
plt.title('Indore July Test Period: SRGAN vs Hybrid Conv1D-MLP vs Benchmark Models', fontsize=13, fontweight='bold')
plt.xlabel('Date'); plt.ylabel('Rainfall (mm)')
plt.xticks(df_july_test['date'], df_july_test['date'].dt.strftime('%d-%b'), rotation=45)
plt.grid(True, alpha=0.3); plt.legend(loc='upper right')
plt.tight_layout(); plt.show()

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import ee
import pandas as pd

# 1. Authenticate and Initialize Earth Engine
# Run ee.Authenticate() if this is your first time in this environment
try:
    ee.Initialize(project='btp-flood-forcast-system')
except Exception as e:
    ee.Authenticate()
    ee.Initialize(project='btp-flood-forcast-system')

def extract_ward_population():
    print("🌍 Fetching Population Data from Google Earth Engine...")

    # 2. Load the GHSL Population Dataset (100m resolution)
    # GHSL provides data for specific epochs. We'll grab the 2020 image (or use 2025 projection).
    pop_collection = ee.ImageCollection("JRC/GHSL/P2023A/GHS_POP")
    pop_image = pop_collection.filterDate('2020-01-01', '2020-12-31').first().select('population_count')

    # Alternative: WorldPop
    # pop_collection = ee.ImageCollection("WorldPop/GP/100m/pop")
    # pop_image = pop_collection.filter(ee.Filter.eq('country', 'IND')).filterDate('2020-01-01', '2020-12-31').first().select('population')

    # 3. Load Indore Wards Boundary FeatureCollection
    # REPLACE THIS with your actual uploaded GEE Asset ID for Indore Wards
    asset_id = "projects/your-project-id/assets/indore_wards_boundary"
    try:
        indore_wards = ee.FeatureCollection(asset_id)
    except Exception as e:
        print(f"❌ Error loading Ward boundaries. Ensure your Asset ID is correct. Error: {e}")
        return

    # 4. Reduce Regions: Sum the population pixels falling inside each ward polygon
    # Scale is set to 100m to match the dataset's native resolution
    ward_population = pop_image.reduceRegions(
        collection=indore_wards,
        reducer=ee.Reducer.sum(),
        scale=100
    )

    # 5. Extract data to local Python environment
    features = ward_population.getInfo()['features']

    data = []
    # Replace 'Ward_No' and 'Ward_Name' with the actual column names from your uploaded shapefile
    for feat in features:
        props = feat['properties']
        data.append({
            "Ward_No": props.get('Ward_No', 'N/A'),
            "Ward_Name": props.get('Ward_Name', 'Unknown'),
            "Estimated_Population": int(props.get('sum', 0)) # 'sum' is the output of ee.Reducer.sum()
        })

    # 6. Convert to Pandas DataFrame for easy viewing and export
    df = pd.DataFrame(data)
    df = df.sort_values(by='Ward_No').reset_index(drop=True)

    print("\n✅ Ward-wise Population Extraction Complete:")
    print(df.to_string())

    # Optionally save for your flood prediction model
    df.to_csv("indore_ward_population_exposure.csv", index=False)
    print("\n💾 Saved to 'indore_ward_population_exposure.csv'")

if __name__ == "__main__":
    extract_ward_population()